In [1]:
from IPython.display import IFrame

IFrame("FD_five_pt_stencil.pdf", width=1000, height=800)

## Answer - 7a

In [2]:
import numpy as np

# Matrix D for the internal node of dimension N\times N for uniform grid
def matrix_D(M):
    N = M-1
    D = np.zeros((N,N))
    for i in range(N):
        D[i, i] = 4
        if i != 0:
            D[i, i-1] = -1
        if i != N-1:
            D[i, i+1] = -1
    return D

# Matrix of the formulation [[0, -1], [-1, 0]]
def create_tridiagonal_matrix(M):
    N = M-1
    A = np.zeros((N, N))
    for i in range(N):
        if i != 0:
            A[i, i-1] = -1
        if i != N-1:
            A[i, i+1] = -1
    return A

# Right hand side
def f_rhs(x, y):
    return -(12*x**2*y**5 + 20*x**4*y**3 + 17*np.sin(x*y)*(x**2 + y**2))
    #return -4

# Exact Solution
def exact_solution(x, y):
    return x**4*y**5-17*np.sin(x*y)
    #return x**2+y**2

# Solve the Poisson Problem
def solve_poisson_rectangular_uniform_grid(M):
    # Interval points
    ax, bx = 0, 2
    ay, by = 0, 1
    h = (bx - ax) / M
    Nx = M - 1            # number of interior points for fixed-j in x-direction
    Ny = int((by - ay)/h) - 1  # number of interior points for fixed-i in y-direction

    D = matrix_D(Ny+1)
    I1 = np.eye(Nx)
    I2 = np.eye(Ny)
    # Diagonal part of matrix A
    A1 = np.kron(I1, D)
    # Other part of matrix A
    A2 = np.kron(create_tridiagonal_matrix(Nx+1), I2)
    A = (A1 + A2)/(h**2)
    # Load vector and Dirichlet BCs
    F = np.zeros(Nx * Ny)
    for i in range(Nx):
        for j in range(Ny):
            x = (i+1)*h
            y = (j+1)*h
            l = i*Ny + j
            F[l] = f_rhs(x, y)

            # Boundary contributions
            if i == 0:
                F[l] += exact_solution(ax, y) / h**2
            if i == Nx - 1:
                F[l] += exact_solution(bx, y) / h**2
            if j == 0:
                F[l] += exact_solution(x, ay) / h**2
            if j == Ny - 1:
                F[l] += exact_solution(x, by) / h**2

    # Solve system
    U_vec = np.linalg.solve(A, F)

    # Reshape to 2D grid
    U_grid = U_vec.reshape((Nx, Ny))

    # Exact solution
    U_exact = np.zeros((Nx, Ny))
    for i in range(Nx):
        for j in range(Ny):
            x = (i+1)*h
            y = (j+1)*h
            U_exact[i, j] = exact_solution(x, y)

    # Infinity norm of error
    error_inf = np.max(np.abs(U_grid - U_exact))

    return h, error_inf

# Mesh sizes to test
mesh_sizes = [4, 8, 16, 32, 64, 128]
error = []
step_size = []

for M in mesh_sizes:
    h, err_inf= solve_poisson_rectangular_uniform_grid(M)
    step_size.append(h)
    error.append(err_inf)

for i in range(len(step_size)):
    if i == 0:
        eoc = 0
    else:
        eoc = np.log(error[i-1] / error[i]) / np.log(step_size[i-1] / step_size[i])
    print(f"h : {step_size[i]:.5f}, Error : {error[i]:.7f}, EOC : {eoc:.5f}")

h : 0.50000, Error : 0.3632489, EOC : 0.00000
h : 0.25000, Error : 0.1080426, EOC : 1.74936
h : 0.12500, Error : 0.0319699, EOC : 1.75681
h : 0.06250, Error : 0.0081049, EOC : 1.97985
h : 0.03125, Error : 0.0020374, EOC : 1.99207
h : 0.01562, Error : 0.0005103, EOC : 1.99733


## Answer - 7b

In [3]:
# Matrix D for the internal node of dimension N\times N for non-uniform grid
def matrix_D_dash(M, hx, hy):
    N = M-1
    D = np.zeros((N,N))
    for i in range(N):
        D[i, i] = 2*( 1/hx**2 + 1/hy**2 )
        if i != 0:
            D[i, i-1] = -1/hy**2
        if i != N-1:
            D[i, i+1] = -1/hy**2
    return D


# Solve the Poisson Problem
def solve_poisson_rectangular_non_uniform_grid(M):
    # Interval points
    ax, bx = 0, 2
    ay, by = 0, 1
    hx = (bx - ax) / M[0]
    hy = (by - ay) / M[1]
    Nx = M[0] - 1              # number of interior points for fixed-j in x-direction
    Ny = M[1] - 1              # number of interior points for fixed-i in y-direction

    D = matrix_D_dash(Ny+1, hx, hy)
    I1 = np.eye(Nx)
    I2 = (1/hx**2)*np.eye(Ny)
    # Diagonal part of matrix A
    A1 = np.kron(I1, D)
    # Other part of matrix A
    A2 = np.kron(create_tridiagonal_matrix(Nx+1), I2)
    A = A1 + A2
    # Load vector and Dirichlet BCs
    F = np.zeros(Nx * Ny)
    for i in range(Nx):
        for j in range(Ny):
            x = (i+1)*hx
            y = (j+1)*hy
            l = i*Ny + j
            F[l] = f_rhs(x, y)

            # Boundary contributions
            if i == 0:
                F[l] += exact_solution(ax, y) / hx**2
            if i == Nx - 1:
                F[l] += exact_solution(bx, y) / hx**2
            if j == 0:
                F[l] += exact_solution(x, ay) / hy**2
            if j == Ny - 1:
                F[l] += exact_solution(x, by) / hy**2

    # Solve system
    U_vec = np.linalg.solve(A, F)

    # Reshape to 2D grid
    U_grid = U_vec.reshape((Nx, Ny))

    # Exact solution
    U_exact = np.zeros((Nx, Ny))
    for i in range(Nx):
        for j in range(Ny):
            x = (i+1)*hx
            y = (j+1)*hy
            U_exact[i, j] = exact_solution(x, y)

    # Infinity norm of error
    error_inf = np.max(np.abs(U_grid - U_exact))

    return hx, hy, error_inf

# Each entry is mesh size in x and y direction
mesh_sizes = [[5,10], [10,15], [20,33], [50,40], [100,77]]

for M in mesh_sizes:
    h_x, h_y, err_inf= solve_poisson_rectangular_non_uniform_grid(M)
    print(f"hx : {h_x:.5f}, hy : {h_y:.5f}, Error : {err_inf:.7f}")

hx : 0.40000, hy : 0.10000, Error : 0.0189038
hx : 0.20000, hy : 0.06667, Error : 0.0090111
hx : 0.10000, hy : 0.03030, Error : 0.0019211
hx : 0.04000, hy : 0.02500, Error : 0.0013069
hx : 0.02000, hy : 0.01299, Error : 0.0003532


## Answer - 7c

In [4]:
from scipy.sparse import lil_matrix
from scipy.sparse.linalg import spsolve


# Exact solution for Question 7(c)
def exact_solution_L(x, y):
    return np.sin(np.pi * x) * np.cos(np.pi * y)


# Right-hand side:
# -Delta u = f
# u = sin(pi*x) cos(pi*y)
#
# Delta u = -2*pi^2*sin(pi*x)cos(pi*y)
# Therefore
# f = 2*pi^2*sin(pi*x)cos(pi*y)
def f_rhs_L(x, y):
    return 2 * np.pi**2 * np.sin(np.pi * x) * np.cos(np.pi * y)


def solve_poisson_L_shape(M):
    # ----------------------------------------------------------
    # Domain:
    #
    # Omega = [0,1]^2 \ [0.5,1] x [0.5,1]
    #
    # ----------------------------------------------------------
    ax, bx = 0.0, 1.0
    ay, by = 0.0, 1.0

    # Interior boundary of the L-shaped domain
    ax1, ay1 = 0.5, 0.5

    # Grid spacing
    hx = (bx - ax) / M[0]
    hy = (by - ay) / M[1]

    Mx = M[0]
    My = M[1]

    # ----------------------------------------------------------
    # Identify all interior unknown points of the L-shaped
    # domain.
    #
    # The removed square is
    #
    #       [0.5,1] x [0.5,1]
    #
    # Hence an interior point belongs to the L-domain if
    #
    #       x < 0.5 or y < 0.5
    # ----------------------------------------------------------

    index = {}
    unknown_points = []

    for i in range(1, Mx):
        for j in range(1, My):

            x = i * hx
            y = j * hy

            inside_L = (x < ax1) or (y < ay1)

            if inside_L:
                k = len(unknown_points)
                index[(i, j)] = k
                unknown_points.append((i, j))

    N = len(unknown_points)

    # ----------------------------------------------------------
    # Construct A U = F
    # ----------------------------------------------------------

    A = lil_matrix((N, N))
    F = np.zeros(N)

    cx = 1.0 / hx**2
    cy = 1.0 / hy**2

    for (i, j), row in index.items():

        x = i * hx
        y = j * hy

        # Central coefficient
        A[row, row] = 2.0 * (cx + cy)

        # RHS
        F[row] = f_rhs_L(x, y)

        # ------------------------------------------------------
        # Five-point stencil
        #
        # 2(1/hx^2 + 1/hy^2) u_ij
        # - u_{i+1,j}/hx^2
        # - u_{i-1,j}/hx^2
        # - u_{i,j+1}/hy^2
        # - u_{i,j-1}/hy^2
        # = f_ij
        #
        # If the neighboring point is another unknown,
        # put the coefficient into A.
        #
        # If the neighboring point lies on the boundary,
        # move its known Dirichlet value to F.
        # ------------------------------------------------------

        neighbors = [
            (i + 1, j, cx),
            (i - 1, j, cx),
            (i, j + 1, cy),
            (i, j - 1, cy)
        ]

        for ni, nj, coefficient in neighbors:

            if (ni, nj) in index:

                # Neighbor is an interior unknown
                col = index[(ni, nj)]
                A[row, col] = -coefficient

            else:

                # Neighbor is on the boundary of the L-shaped
                # domain. Its value is known from the Dirichlet BC.
                xn = ni * hx
                yn = nj * hy

                F[row] += coefficient * exact_solution_L(xn, yn)

    # ----------------------------------------------------------
    # Solve the sparse linear system
    # ----------------------------------------------------------

    U_vec = spsolve(A.tocsr(), F)

    # ----------------------------------------------------------
    # Compute infinity norm of error
    # ----------------------------------------------------------

    error_inf = 0.0

    for (i, j), k in index.items():

        x = i * hx
        y = j * hy

        error = abs(
            U_vec[k] - exact_solution_L(x, y)
        )

        error_inf = max(error_inf, error)

    return hx, hy, error_inf


# --------------------------------------------------------------
# Convergence test
#
# IMPORTANT:
# Mx and My should be even so that x=0.5 and y=0.5 lie
# exactly on grid lines.
# --------------------------------------------------------------

mesh_sizes = [
    [10, 10],
    [20, 20],
    [40, 40],
    [80, 80],
    [160, 160]
]

error = []
step_size = []

for M in mesh_sizes:

    hx, hy, err_inf = solve_poisson_L_shape(M)

    error.append(err_inf)
    step_size.append(hx)

    print(
        f"Mx = {M[0]:3d}, "
        f"My = {M[1]:3d}, "
        f"hx = {hx:.5f}, "
        f"hy = {hy:.5f}, "
        f"Error = {err_inf:.7e}"
    )


# --------------------------------------------------------------
# EOC
# --------------------------------------------------------------

print("\nConvergence table:\n")

for i in range(len(error)):

    if i == 0:
        eoc = 0.0
    else:
        eoc = np.log(error[i-1] / error[i]) / np.log(
            step_size[i-1] / step_size[i]
        )

    print(
        f"h = {step_size[i]:.5f}, "
        f"Error = {error[i]:.7e}, "
        f"EOC = {eoc:.5f}"
    )

Mx =  10, My =  10, hx = 0.10000, hy = 0.10000, Error = 2.8108715e-03
Mx =  20, My =  20, hx = 0.05000, hy = 0.05000, Error = 7.0597506e-04
Mx =  40, My =  40, hx = 0.02500, hy = 0.02500, Error = 1.7760898e-04
Mx =  80, My =  80, hx = 0.01250, hy = 0.01250, Error = 4.4476318e-05
Mx = 160, My = 160, hx = 0.00625, hy = 0.00625, Error = 1.1128331e-05

Convergence table:

h = 0.10000, Error = 2.8108715e-03, EOC = 0.00000
h = 0.05000, Error = 7.0597506e-04, EOC = 1.99333
h = 0.02500, Error = 1.7760898e-04, EOC = 1.99091
h = 0.01250, Error = 4.4476318e-05, EOC = 1.99760
h = 0.00625, Error = 1.1128331e-05, EOC = 1.99880
